# Embedding significant results
### Author: Oscar Heath
### Date: Sept 12 2026

The file significant_results.parquet contains 800 MWAS results for *totiviridiae* which have corrected p-value < 0.05. We now format the results into a structured string, and embed them using the `all-MiniLM-L6-v2` sentence transformer. We then store, for each finding, the top `k` processed restuls with the closest embeddings (`k` controlled by `BLOCKING_PARAM` variable)

In [2]:
import polars as pl
from sentence_transformers import SentenceTransformer

c:\Users\oscar\Documents\totiviridiae_mwas_analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
BLOCKING_PARAM = 3

In [4]:
df = pl.read_parquet("../data/significant_results.parquet")

def process_row(row):
    col_names = row["col_name"]
    values = row["col_value"]
    attributes = [f"{col}={val}" for col, val in zip(col_names, values)]
    results = f"Samples with the following attributes: {', '.join(attributes)} had a mean concentration of totiviridiae of {row['mean_positive']} and a mean concentration of totiviridiae of {row['mean_negative']} in the negative samples."
    return results

# add a new column to the dataframe with the processed results
df = df.with_columns(
    pl.struct(["col_name", "col_value", "mean_positive", "mean_negative"])
    .map_elements(process_row, return_dtype=pl.String)
    .alias("processed_result")
)

In [5]:
df.write_parquet("../data/processed_results.parquet")

In [6]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
sentences = df["processed_result"].to_list()
embeddings = model.encode(sentences, convert_to_tensor=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4137.35it/s]


In [7]:
similarities = model.similarity(embeddings, embeddings)


In [8]:
import torch

# exclude self-similarity so a row's own result never appears in its own top matches
n = similarities.shape[0]
self_mask = torch.eye(n, dtype=torch.bool)
similarities_no_self = similarities.masked_fill(self_mask, float("-inf"))

top_k_indices = similarities_no_self.topk(BLOCKING_PARAM, dim=1).indices

processed_results = df["processed_result"].to_list()
top_k_indices_list = top_k_indices.tolist()
top_k_results = [
    [processed_results[idx] for idx in row_indices]
    for row_indices in top_k_indices_list
]

df = df.with_columns(
    pl.Series("top_similar_results", top_k_results),
    pl.Series("top_similar_indices", top_k_indices_list),
)

In [9]:
df.write_parquet("../data/processed_results.parquet")

In [ ]:
# generate a df of pairs of processed results and their top similar results
pairs = []
df_indexed = df.with_row_index("index")

for row in df_indexed.iter_rows(named=True):
    for j in row["top_similar_indices"]:
        pairs.append(
            {
                "row_1": row["index"],
                "row_2": j,
                "result_1": row["processed_result"],
                "result_2": df_indexed["processed_result"][j],
            }
        )
pairs_df = pl.DataFrame(pairs)
pairs_df.write_parquet("../data/pairs.parquet")